# 01 — Dataset download, validation, and few-shot preparation

**Purpose:** Download the official SIB-200 English/Urdu splits, preserve source IDs, detect leakage, and create repeatable nested training subsets.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# Kaggle/Colab project setup — NO PIP INSTALLATIONS.
# Uses the versions already installed in this runtime.
from pathlib import Path
from importlib import metadata
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PULL_LATEST = False  # Enable only for a clean clone after pushing fixes to GitHub.
if Path("/kaggle/working").is_dir():
    WORKSPACE = Path("/kaggle/working")
elif Path("/content").is_dir():
    WORKSPACE = Path("/content")
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / "nlp_research_proj"
ROOT = REPO_DIR / "Urdu_Qwen_Research_Notebooks"

if (REPO_DIR / ".git").is_dir():
    if PULL_LATEST:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git folder: {REPO_DIR}; inspect before removing it")
elif (Path.cwd() / "project_utils.py").is_file():
    ROOT = Path.cwd()
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(f"Project code missing from {ROOT}")

# Check user-confirmed Kaggle versions; NEVER uninstall or upgrade preinstalled dependencies.
EXPECTED = {
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "torch": "2.11.0",  # the Kaggle build has a +cu128 suffix
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
}
versions = {}
for name, expected in EXPECTED.items():
    try:
        got = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Missing preinstalled dependency: {name}") from None
    versions[name] = got
    if got.split("+")[0] != expected:
        raise RuntimeError(
            f"This code is checked for {name}={expected}, found {got}. "
            "Do NOT run pip; use the tested Kaggle image or review the API changes."
        )
for name in ("accelerate", "datasets", "pandas", "matplotlib", "pytest"):
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Required package missing: {name}. This runtime cannot run the selected notebook.") from None
print("Using preinstalled Kaggle libraries (no installations):", versions)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
api = check_training_api()
print("Project ready:", ROOT)
print("Training API ready:", api["transformers"], api["trainer_tokenizer_key"], api["warmup_key"])


**Rules:** Do not alter or resplit the official test data; do not use the paper's 1,000 test headlines for training. Keep Urdu and English labels identical, save source IDs, and verify that no corresponding sentence appears in different splits. Datasets are downloaded from [Davlan/SIB-200](https://huggingface.co/datasets/Davlan/sib200).

In [ ]:
from datasets import load_dataset
from collections import Counter
TASK = 'sib200'
LANGS = {'ur': 'urd_Arab', 'en': 'eng_Latn'}
DATA = {}
for lang, lang_id in LANGS.items():
    DATA[lang] = load_dataset('Davlan/sib200', lang_id)
    print(lang, {s: len(DATA[lang][s]) for s in DATA[lang]})
    print('fields:', DATA[lang]['train'].column_names)
    assert {'index_id', 'category', 'text'} <= set(DATA[lang]['train'].column_names)
    actual_categories = set(DATA[lang]['train']['category'])
    print('Categories:', sorted(actual_categories))
    assert actual_categories == set(DEFAULT_LABELS), f'Unexpected category names: {actual_categories}'


In [ ]:
# SIB-200 uses string `category` values rather than a numeric `label` feature.
labels = list(DEFAULT_LABELS)
label_to_id = {name: index for index, name in enumerate(labels)}
rows = []
for lang, dsd in DATA.items():
    for split in ['train', 'validation', 'test']:
        for r in dsd[split]:
            label = str(r['category'])
            assert label in label_to_id, f'Unexpected label: {label}'
            item = dict(example_id=stable_id(TASK, lang, split, str(r['index_id'])),
                        index_id=str(r['index_id']), source='Davlan/sib200', lang=lang, split=split,
                        text=str(r['text']), label_id=label_to_id[label], label=label)
            rows.append(item)
problems = validate_split(rows, labels)
print('Rows:', len(rows), 'issues:', problems[:10])
assert not problems, 'Inspect split leakage, missing values or duplicate IDs before proceeding.'
for s in ['train', 'validation', 'test']:
    en = {r['index_id']: r['label_id'] for r in rows if r['lang']=='en' and r['split']==s}
    ur = {r['index_id']: r['label_id'] for r in rows if r['lang']=='ur' and r['split']==s}
    assert en == ur, 'Parallel English and Urdu IDs/labels do not match; inspect before continuing.'
print('Parallel languages and labels match by index_id across all splits.')


In [ ]:
out = ROOT/'data'/TASK; out.mkdir(exist_ok=True)
for lang in LANGS:
    for split in ['train','validation','test']:
        group = [r for r in rows if r['lang']==lang and r['split']==split]
        write_csv(out / f'{lang}_{split}.csv', group)
counts = {lang: {s:Counter(r['label'] for r in rows if r['lang']==lang and r['split']==s)
                 for s in ['train','validation','test']} for lang in LANGS}
manifest = {'task':TASK, 'dataset':'Davlan/sib200','language_configs':LANGS, 'labels':labels,
            'rows_by_split': {f'{lang}_{split}':len([r for r in rows if r['lang']==lang and r['split']==split])
                              for lang in LANGS for split in ['train','validation','test']},
            'sha256':{f'{lang}_{split}':file_hash(out/f'{lang}_{split}.csv')
                      for lang in LANGS for split in ['train','validation','test']},
            'notes':'Fixed official splits; class mapping fixed to DEFAULT_LABELS, dataset has string category feature. Do not change test labels.'}
write_json(out/'manifest.json',manifest)
print('Saved:',out)
for lang in LANGS:
    print(lang,'class counts:',{split:dict(c) for split,c in counts[lang].items()})

In [ ]:
for seed in [42,123,456]:
    subsets = stratified_prefix_samples(load_task(TASK,'ur','train'), [5,10,25,50,100], seed=seed)
    for n, entry in subsets.items():
        if entry['available']:
            write_csv(out / f'ur_train_{n}perclass_seed{seed}.csv', entry['rows'])
            print(f'seed={seed} n/class={n} total={len(entry["rows"])}')
        else:
            print(f'seed={seed} n/class={n} NOT available:', entry['shortfall'])
print('Important: 100 examples per class is not guaranteed; do not oversample without explicitly reporting it.')